# Food Trust -- Product to WHO NPPM Category Mapping (Epic 3)

Builds `product_nppm_category_mapping.db`, a rule-based classification of
every product in `products.db` onto one of WHO NPPM 2022's 13 product
subcategories from `who_nppm_thresholds.db`. This is the missing link
that lets Epic 3's US3.1 comparison cards actually pick the right
threshold row for a real product.

**Read the whole build cell's docstring below before wiring this into
the backend.** This is a best-effort, rule-based mapping built from the
real distribution of `categories_tags`/`labels_tags` actually present in
the 7036 products, not a guaranteed-correct classifier -- OFF's tags were
designed for general product browsing, not for WHO NPPM's specific
nutrition-policy category boundaries, some of which depend on the exact
wording and word order of the product name on the pack (categories 4.3
vs 4.4 in particular). Every rule below is checked against real counts
in this notebook's own output, and anything the rules cannot answer
confidently is marked `unclassified` rather than guessed.

## Build product_nppm_category_mapping.db

In [1]:

"""
Builds product_nppm_category_mapping.db: a best-effort, rule-based mapping
of every product in products.db onto one of WHO NPPM 2022's 13 product
subcategories (see who_nppm_thresholds.db), so Epic 3's US3.1 red/green
cards know which threshold row to compare a given product against.

This is NOT a finished, guaranteed-correct classifier. Open Food Facts
categories_tags/labels_tags were designed for a different purpose (general
product browsing/search) than WHO NPPM's 13 categories (a nutrition-policy
scheme built around exactly what a product is composed of and, in several
cases, the literal word order of the product name on the pack). The rules
below were built by inspecting the real distribution of tags actually
present across the 7036 products in products.db, not guessed abstractly,
and every rule is checked against real counts in this build's own output.
Where the source data genuinely cannot support a confident answer, a
product is marked "unclassified" or given a "low"/"medium" confidence
tag rather than forced into a guess.

Read this docstring fully before wiring this into the backend -- the
open questions at the end of the build output are not edge cases to
patch later, they are structural limits of what OFF's tags can tell us.
"""

import sqlite3, json, re, collections, shutil, tempfile
from pathlib import Path

BASE_DIR = Path.cwd()
local_products = Path(tempfile.mkdtemp()) / "products.db"
shutil.copy(BASE_DIR / "products.db", local_products)
src = sqlite3.connect(local_products)
src.row_factory = sqlite3.Row

# ---------------------------------------------------------------------
# STEP 0 -- out of NPPM scope entirely
# ---------------------------------------------------------------------
# WHO NPPM Table 1 governs commercial complementary FOODS for 6-36
# months. Infant formula / follow-on / growth milk is regulated under a
# completely separate WHO framework (the Code of Marketing of
# Breast-Milk Substitutes), not NPPM -- and NPPM Table 1 itself
# explicitly EXCLUDES "breastmilk substitutes or unsweetened milk/milk
# alternatives" from category 8 (Drinks) too, so formula/growth milk
# does not belong in ANY of the 13 categories. This is a real, large
# slice of products.db: en:baby-milks alone is 1356 of the 7036 rows.
FORMULA_MILK_TAGS = {
    "en:baby-milks", "en:infant-formulas", "en:growth-milks",
    "en:milk-powders", "en:first-age-baby-milk-powder",
    "en:second-age-baby-milk-powder", "en:ready-to-feed-baby-milks",
    "en:ready-to-feed-baby-growing-up-milk",
    "en:ready-to-feed-baby-first-milk",
    "en:ready-to-feed-baby-follow-on-milk",
    "en:baby-follow-on-milk-from-5-months",
    "en:baby-follow-on-milk-from-2-years",
    "en:baby-start-milk-from-10-months",
    "en:milks-liquid-and-powder",
    "nl:zuigelingenmelk", "nl:Zuigelingenmelk",
    "nl:opvolgmelk", "nl:Opvolgmelk",
    "nl:flesvoeding", "nl:Flesvoeding",
}

# ---------------------------------------------------------------------
# STEP 1 -- category 8 Drinks (excluded from promotion), with the
# puree/smoothie exception NPPM states explicitly in its own text
# ---------------------------------------------------------------------
PUREE_FORMAT_EXCEPTION_TAGS = {
    "en:compotes-to-drink", "en:smoothies", "en:fruit-smoothies",
}
DRINK_TAGS = {
    "en:beverages", "en:baby-drinks", "en:juices-and-nectars",
    "en:fruit-juices", "en:specialised-baby-drinks",
    "en:sweetened-beverages", "en:unsweetened-beverages",
    "en:fermented-drinks", "en:fermented-milk-drinks",
    "en:dairy-drinks", "en:plant-based-beverages",
    "en:milky-cereal-based-beverages", "en:cereal-based-drinks",
    "en:dehydrated-beverages", "en:beverages-and-beverages-preparations",
    "en:fruit-based-beverages", "en:plant-based-beverages-for-babies",
    "en:fruit-based-beverages-for-babies",
}

# ---------------------------------------------------------------------
# STEP 2 -- category 7 Confectionery (excluded from promotion)
# Deliberately narrow: only genuinely chocolate/cocoa confectionery
# tags. en:chocolate-biscuits and en:sweet-snacks stay in the 5.2
# snack bucket below (a biscuit is a cereal-based snack per NPPM's own
# category 5.2 definition even if chocolate-coated) -- flagged as a
# judgment call in the summary, not hard-coded as confectionery.
# ---------------------------------------------------------------------
CONFECTIONERY_TAGS = {
    "en:chocolate-desserts", "en:cocoa-and-its-products",
}

# ---------------------------------------------------------------------
# STEP 3 -- category 2 Dairy foods/desserts/cereals vs 3.1 fruit-
# containing. NPPM's own rule: dairy dessert stays "2" only if fruit
# content is <=5%; above that it is "3.1". Tags cannot tell us an
# actual percentage, so a dairy product that is ALSO fruit-tagged is
# marked medium confidence, defaulting to 3.1 (a product whose fruit
# content is called out by name/tag is more likely to be a real
# fruit-forward recipe than an incidental <=5% garnish).
# ---------------------------------------------------------------------
DAIRY_TAGS = {
    "en:dairy-dessert-for-baby", "en:yogurts",
    "en:fermented-dairy-desserts", "en:dairy-desserts",
    "en:plain-dairy-dessert-for-baby", "en:creamy-puddings",
    "en:stirred-yogurts", "en:dairy-dessert-for-baby-with-semolina",
    "en:dairy-dessert-for-baby-with-sugar",
    "en:fermented-milk-products", "en:plain-fermented-dairy-desserts",
}
DAIRY_FRUIT_TAGS = {
    "en:fruit-yogurts", "en:fermented-dairy-desserts-with-fruits",
    "en:plain-dairy-dessert-for-baby-with-fruits",
}

# ---------------------------------------------------------------------
# STEP 4 -- category 3.1 Fruit-containing product
# ---------------------------------------------------------------------
FRUIT_TAGS = {
    "en:compotes", "en:baby-fruit-desserts", "en:apple-compotes",
    "en:fruits-puree-without-sugar-added", "en:fruit-based-foods",
    "en:pear-compotes", "en:banana-apple-compotes", "en:applesauces",
    "en:fruits-desserts", "en:mango-applesauces", "en:peach-applesauce",
    "en:fruit-based-foods-and-beverages", "en:fruits-based-foods",
    "en:fruits", "en:compotes-to-drink", "en:smoothies",
    "en:fruit-smoothies", "fr:compotes-pour-bebe", "fr:compotes-sans-sucres",
}

# ---------------------------------------------------------------------
# STEP 5 -- category 3.2 Vegetable-only product
# Only fires if the product is NOT also a starch/meat/fish combination
# tag (those go to 4.x below -- checked first in the classify() order).
# ---------------------------------------------------------------------
VEG_ONLY_TAGS = {
    "en:mashed-vegetables", "en:vegetables-based-foods",
    "en:vegetable-based-foods-and-beverages", "en:vegetables",
    "en:carrots", "en:prepared-vegetables",
    "en:baby-food-jar-with-vegetables-from-4-6-months",
}

# ---------------------------------------------------------------------
# STEP 6 -- category 4.1-4.5 Savoury meals
# ---------------------------------------------------------------------
SAVORY_MEAL_TAGS = {
    "en:main-meals-for-babies", "en:baby-dishes",
    "en:evening-meals-for-babies", "en:meat-menus-for-babies",
    "en:fish-menus-for-babies", "en:vegetables-menus-for-babies",
    "en:beef-dishes", "en:pasta-dishes", "en:rice-dishes",
    "en:bolognese-style-pasta", "en:bolognese-spaghetti",
    "en:shepherd-s-pie", "en:soups-for-babies", "en:canned-meals",
    "en:microwave-meals", "en:poultry-meals", "en:meals-with-meat",
    "en:meals-with-fish", "en:meals",
    "en:baby-food-jar-with-vegetables-and-starch-from-4-6-months",
}
SAVORY_MEAL_SUBSTRINGS = ("vegetable-dish-for-bab",)

PROTEIN_KEYWORDS = [
    "chicken", "beef", "turkey", "fish", "salmon", "lamb", "pork",
    "veal", "ham", "poultry", "tuna", "cod", "shrimp",
    "poulet", "boeuf", "dinde", "poisson", "saumon", "agneau", "porc",
    "veau", "jambon", "volaille", "thon", "cabillaud",
    "kip", "rund", "kalkoen", "vis", "zalm", "lam", "varken", "kalf",
    "huhn", "rind", "pute", "fisch", "lachs", "lamm", "schwein", "kalb",
]
CHEESE_KEYWORDS = ["cheese", "fromage", "kaas", "kase", "kase"]

# ---------------------------------------------------------------------
# STEP 7 -- category 5.1 Fruit snacks (no dedicated OFF tag found in
# this dataset -- kept as an explicit, empty-by-tag-rule bucket; would
# need a name/ingredients keyword search (dried fruit, raisins, etc.)
# to populate, not attempted here -- flagged as a known gap)
# ---------------------------------------------------------------------

# ---------------------------------------------------------------------
# STEP 8 -- category 5.2 Dry/semi-dry snacks and finger foods
# ---------------------------------------------------------------------
SNACK_5_2_TAGS = {
    "en:biscuits-for-babies", "en:biscuits-and-cakes", "en:biscuits",
    "en:puffed-cereal-cakes", "en:puffed-rice-cakes", "en:cereal-bars",
    "en:bars", "en:dry-biscuits", "en:crackers-appetizers",
    "en:sponge-fingers-biscuit", "en:salty-snacks", "en:sweet-snacks",
    "en:chocolate-biscuits", "en:appetizers",
}

# ---------------------------------------------------------------------
# STEP 9 -- category 6 Ingredients
# ---------------------------------------------------------------------
INGREDIENT_TAGS = {
    "en:seeds", "en:cereal-grains", "en:spreads", "en:condiments",
    "en:cheeses",
}

# ---------------------------------------------------------------------
# STEP 10 -- category 1 Dry cereals and starches
# ---------------------------------------------------------------------
CEREAL_STARCH_TAGS = {
    "en:cereals-for-babies", "en:breakfast-cereals", "en:porridge",
    "en:rolled-flakes", "en:flakes", "en:cereal-flours",
    "en:cereal-semolinas", "en:flours",
}
CEREAL_STARCH_SUBSTRINGS = ("instant-cereal-powder-to-be-reconstituted",)


def first_two_tokens(name):
    return re.findall(r"[a-zA-Z]+", (name or "").lower())[:2]


def classify(tags, labels, name):
    tagset = set(tags or [])
    labelset = set(labels or [])
    name_lower = (name or "").lower()

    if tagset & FORMULA_MILK_TAGS:
        return ("out_of_scope", "high", "infant formula / growth milk -- not covered by NPPM Table 1 at all")

    if tagset & PUREE_FORMAT_EXCEPTION_TAGS:
        return ("3.1", "medium", "drinkable-format puree/smoothie -- NPPM explicitly excludes purees from category 8 regardless of format")

    if tagset & DRINK_TAGS:
        return ("8", "high", "matched a Drinks tag; category 8 is excluded from promotion")

    if tagset & CONFECTIONERY_TAGS:
        return ("7", "medium", "matched a chocolate/cocoa dessert tag; category 7 is excluded from promotion")

    is_savory = bool(tagset & SAVORY_MEAL_TAGS) or any(
        sub in t for t in tagset for sub in SAVORY_MEAL_SUBSTRINGS
    )
    if is_savory:
        has_cheese = any(k in name_lower for k in CHEESE_KEYWORDS)
        has_protein = any(k in name_lower for k in PROTEIN_KEYWORDS)
        if has_protein:
            first_two = first_two_tokens(name)
            named_first = any(
                any(k == tok or k in tok for k in PROTEIN_KEYWORDS)
                for tok in first_two
            )
            if named_first:
                return ("4.4", "medium", "savoury meal, protein keyword in first two words of name")
            return ("4.3", "medium", "savoury meal, protein keyword present but not in first two words of name")
        if has_cheese:
            return ("4.2", "medium", "savoury meal, cheese keyword in name, no protein keyword found")
        return ("4.1", "medium", "savoury meal tag, no protein/cheese keyword found in name")

    if tagset & DAIRY_FRUIT_TAGS:
        return ("3.1", "medium", "dairy dessert tagged with a fruit variant -- NPPM routes >5% fruit dairy desserts to 3.1, defaulted here since exact % is unknown")

    if tagset & DAIRY_TAGS:
        return ("2", "medium", "matched a plain dairy dessert/yogurt tag with no fruit tag")

    if tagset & FRUIT_TAGS:
        return ("3.1", "high", "matched a fruit-containing product tag")

    if tagset & VEG_ONLY_TAGS:
        return ("3.2", "medium", "matched a vegetable-only tag with no starch/meat/fish combination tag")

    if tagset & SNACK_5_2_TAGS:
        return ("5.2", "medium", "matched a dry/semi-dry snack tag")

    if tagset & INGREDIENT_TAGS:
        return ("6", "low", "matched a generic ingredient tag -- could also be a component of a dish rather than a standalone ingredient product")

    if (tagset & CEREAL_STARCH_TAGS) or any(
        sub in t for t in tagset for sub in CEREAL_STARCH_SUBSTRINGS
    ):
        return ("1", "medium", "matched a dry cereal/starch tag")

    return ("unclassified", "none", "no rule matched any categories_tags entry for this product")


rows = src.execute(
    "SELECT code, name, categories_tags, labels_tags FROM products"
).fetchall()

results = []
for r in rows:
    tags = json.loads(r["categories_tags"]) if r["categories_tags"] else []
    labels = json.loads(r["labels_tags"]) if r["labels_tags"] else []
    category_code, confidence, reason = classify(tags, labels, r["name"])
    results.append(dict(
        code=r["code"], category_code=category_code,
        confidence=confidence, match_reason=reason,
    ))

print(f"Classified {len(results)} products.")
print()
print("Counts per category_code:")
counts = collections.Counter(x["category_code"] for x in results)
for code, cnt in sorted(counts.items(), key=lambda kv: -kv[1]):
    pct = cnt / len(results) * 100
    print(f"  {code:<14}{cnt:6d}  ({pct:5.1f}%)")
print()
print("Counts by confidence tier:")
conf_counts = collections.Counter(x["confidence"] for x in results)
for tier, cnt in sorted(conf_counts.items(), key=lambda kv: -kv[1]):
    print(f"  {tier:<8}{cnt:6d}")
print()
print("Sample of 3 product names per category (sanity check):")
by_cat = collections.defaultdict(list)
for r, res in zip(rows, results):
    by_cat[res["category_code"]].append(r["name"])
for code in sorted(by_cat, key=lambda c: (c in ("unclassified", "out_of_scope"), c)):
    names = [n for n in by_cat[code] if n][:3]
    print(f"  [{code}] " + " | ".join(names))

local_tmp_db = Path(tempfile.mkdtemp()) / "product_nppm_category_mapping.db"
out = sqlite3.connect(local_tmp_db)
cur = out.cursor()
cur.execute("""
CREATE TABLE product_nppm_category_mapping (
    code TEXT PRIMARY KEY,
    category_code TEXT NOT NULL,
    confidence TEXT NOT NULL,
    match_reason TEXT NOT NULL
)
""")
cur.executemany(
    "INSERT INTO product_nppm_category_mapping (code, category_code, confidence, match_reason) VALUES (?, ?, ?, ?)",
    [(x["code"], x["category_code"], x["confidence"], x["match_reason"]) for x in results],
)
out.commit()
out.close()
src.close()

import hashlib
local_hash = hashlib.sha256(local_tmp_db.read_bytes()).hexdigest()
OUT_DB = BASE_DIR / "product_nppm_category_mapping.db"
shutil.copy(local_tmp_db, OUT_DB)
mount_hash = hashlib.sha256(Path(OUT_DB).read_bytes()).hexdigest()
print()
print("Copied to:", OUT_DB)
print("local build sha256:", local_hash)
print("mount copy sha256: ", mount_hash)
print("MATCH" if local_hash == mount_hash else "MISMATCH -- DO NOT TRUST THIS FILE")


Classified 7036 products.

Counts per category_code:
  unclassified    2961  ( 42.1%)
  out_of_scope    1411  ( 20.1%)
  4.1              721  ( 10.2%)
  3.1              621  (  8.8%)
  1                356  (  5.1%)
  4.3              326  (  4.6%)
  5.2              233  (  3.3%)
  2                177  (  2.5%)
  8                158  (  2.2%)
  4.4               23  (  0.3%)
  3.2               22  (  0.3%)
  6                 11  (  0.2%)
  7                 10  (  0.1%)
  4.2                6  (  0.1%)

Counts by confidence tier:
  none      2961
  high      2114
  medium    1950
  low         11

Sample of 3 product names per category (sanity check):
  [1] Whoel Grain Multi-Grain Cereal | CERELAC, Céréale infantile lactée | Blédine Multicéréales
  [2] Greek yogi’s | Organic yogis freezedried yogurt fruit snacks strawberry | NESTLE P'TIT ONCTUEUX Fromag Blanc Saveur Natur Sucré-4x100gDès6mois
  [3.1] Pears, Mangos & Spinach- Stage 2 | Apples, Spinach & Kale- Stage 2: Simple Comb

## Schema

| column | notes |
|---|---|
| code | product barcode, matches `products.code` |
| category_code | one of WHO NPPM's 13 codes, or `out_of_scope` (infant formula/growth milk, not covered by NPPM at all), or `unclassified` (no rule matched) |
| confidence | `high` / `medium` / `low` / `none` -- see the rule-by-rule notes in the build cell |
| match_reason | which specific tag or heuristic fired, in plain English |

## The most important finding: a large share of products.db is out of NPPM's scope entirely

Infant formula, follow-on formula and growth milk (`en:baby-milks`,
`en:infant-formulas`, `en:growth-milks`, etc.) are regulated under a
different WHO framework (the Code of Marketing of Breast-Milk
Substitutes), not NPPM. NPPM's own Table 1 text explicitly excludes
"breastmilk substitutes or unsweetened milk/milk alternatives" even from
its own Drinks category (8) -- so these products do not belong in any of
the 13 categories, and should not be force-fit into one. See the build
output above for the actual count (`out_of_scope`) -- this is a
substantial fraction of the 7036 products, not a rounding error.

## Known structural limits (not bugs -- these need either a smarter
## heuristic, a different data source, or a team/mentor decision)

1. **Categories 4.3 vs 4.4** (whether the named protein appears first in
   the product name) can only be approximated from the product's `name`
   text, using a small hand-built multi-language keyword list (English/
   French/Dutch/German). This will misfire on names OFF tagging did not
   anticipate (uncommon protein words, translated names, marketing
   phrasing that does not lead with the ingredient). Treat every 4.3/4.4
   result as `medium` confidence, not ground truth.
2. **Category 4.5** (protein is the ONLY named food) was not
   auto-detected at all -- distinguishing "pureed cooked chicken" (4.5)
   from "chicken and rice" (4.4) reliably needs more than a keyword
   check, and this dataset's product naming makes it genuinely rare.
   Flagged as a gap rather than guessed.
3. **Category 5.1** (fruit snacks -- dried fruit pieces) has no
   dedicated OFF tag in this dataset at all, so the rule set has no rows
   for it currently. Would need a keyword search over `name`/
   `ingredients_text` (e.g. "dried apple", "raisins") to populate --
   not attempted in this build.
4. **Category 2 vs 3.1** (dairy dessert vs fruit-containing dessert)
   depends on NPPM's own "<=5% fruit" rule, which categories_tags cannot
   quantify -- a dairy product that is also fruit-tagged is defaulted to
   3.1 (see `match_reason`), which is a judgment call, not a certainty.
5. **Category 4.2** (cheese named, no other protein) is detected via the
   literal word "cheese"/"fromage"/"kaas" in the product name -- a
   savoury-meal tag alone that only mentions "milk" is NOT treated as
   cheese-named, per NPPM's own more specific definition.
6. **Chocolate-coated snacks** (`en:chocolate-biscuits`) were kept in
   the 5.2 snack bucket rather than routed to Confectionery (7) -- NPPM
   defines category 7 as "chocolates, sweets, liquorice, marzipan, fruit
   chews", which a chocolate-coated cereal biscuit is not quite the same
   as, but this is a judgment call worth a second opinion, not a settled
   fact.

## Suggested next step for whoever picks this up

Do not treat `medium`/`low` confidence rows as final. A reasonable
follow-up: sample a stratified set of `medium`/`low`/`unclassified` rows
per category (the way the `en:baby-foods` filter itself was validated
with a 40-product spot check earlier in this project) and either correct
the rule set or hand-correct the sample before this feeds a real user-
facing comparison card.